In [2]:
import pandas as pd
import sys
sys.path.append('../src')

from popularity_recommender import build_popularity_table, recommend_popular

# Load cleaned data
ratings = pd.read_csv('../data/processed/ratings_clean.csv')
movies = pd.read_csv('../data/processed/movies_clean.csv')

# Build and test
popularity_table = build_popularity_table(ratings, movies, min_ratings=500)
top_10 = recommend_popular(popularity_table, n=10)

print(top_10)

   movieId                                          title  \
0      318               Shawshank Redemption, The (1994)   
1      858                          Godfather, The (1972)   
2       50                     Usual Suspects, The (1995)   
3      527                        Schindler's List (1993)   
4     1221                 Godfather: Part II, The (1974)   
5     2019    Seven Samurai (Shichinin no samurai) (1954)   
6      904                             Rear Window (1954)   
7     7502                        Band of Brothers (2001)   
8      912                              Casablanca (1942)   
9      922  Sunset Blvd. (a.k.a. Sunset Boulevard) (1950)   

                    genres  avg_rating  rating_count  
0              Crime|Drama    4.446990         63366  
1              Crime|Drama    4.364732         41355  
2   Crime|Mystery|Thriller    4.334372         47006  
3                Drama|War    4.310175         50054  
4              Crime|Drama    4.275641         27398 

In [3]:
# Compare how the qualifying pool changes with different thresholds
for threshold in [100, 500, 1000, 5000]:
    table = build_popularity_table(ratings, movies, min_ratings=threshold)
    print(f"min_ratings={threshold}: {len(table):,} movies qualify")

min_ratings=100: 8,546 movies qualify
min_ratings=500: 4,489 movies qualify
min_ratings=1000: 3,159 movies qualify
min_ratings=5000: 1,005 movies qualify


In [4]:
sys.path.append('../src')
from content_based_recommender import build_similarity_matrix, recommend_similar_movies

genres_encoded = pd.read_csv('../data/features/genres_encoded.csv')

# Build the similarity matrix (genres only, for now)
genre_similarity = build_similarity_matrix(genres_encoded)

print("Similarity matrix shape:", genre_similarity.shape)

# Test: find movies similar to Toy Story (movieId = 1)
similar_to_toy_story = recommend_similar_movies(
    movie_id=1, similarity_df=genre_similarity, movies=movies, n=10
)
print(similar_to_toy_story)

Similarity matrix shape: (27278, 27278)
   movieId                                              title  \
0   103755                                       Turbo (2013)   
1   115875          Toy Story Toons: Hawaiian Vacation (2011)   
2    53121                             Shrek the Third (2007)   
3   117454                           The Magic Crystal (2011)   
4    45074                                   Wild, The (2006)   
5    33463  DuckTales: The Movie - Treasure of the Lost La...   
6     3754     Adventures of Rocky and Bullwinkle, The (2000)   
7   114552                              Boxtrolls, The (2014)   
8     2294                                        Antz (1998)   
9   131248                              Brother Bear 2 (2006)   

                                        genres  similarity_score  
0  Adventure|Animation|Children|Comedy|Fantasy               1.0  
1  Adventure|Animation|Children|Comedy|Fantasy               1.0  
2  Adventure|Animation|Children|Comedy|Fant

In [7]:
import pandas as pd
import sys
sys.path.append('../src')

from scipy.sparse import load_npz
from content_based_recommender import (
    build_combined_similarity,
    recommend_similar_movies
)

# Load data
movies = pd.read_csv('../data/processed/movies_clean.csv')
genres_encoded = pd.read_csv('../data/features/genres_encoded.csv')
tfidf_matrix = load_npz('../data/features/tfidf_matrix.npz')
tfidf_movie_ids = pd.read_csv('../data/features/tfidf_movieId_index.csv')['movieId']

# Build combined similarity
print("Building combined similarity matrix — may take 1-3 minutes...")
combined_similarity = build_combined_similarity(
    genres_df=genres_encoded,
    tfidf_matrix=tfidf_matrix,
    tfidf_movie_ids=tfidf_movie_ids,
    genre_weight=0.3,
    tag_weight=0.7
)
print("Done. Shape:", combined_similarity.shape)

# Sanity check
print("\nSanity check:")
print("Toy Story vs Toy Story 2:", round(combined_similarity.loc[1, 3114], 4))
print("Toy Story vs Magic Crystal:", round(combined_similarity.loc[1, 117454], 4))

# Recommend
print("\nTop 10 movies similar to Toy Story:")
print(recommend_similar_movies(1, combined_similarity, movies, n=10))

ImportError: cannot import name 'build_combined_similarity' from 'content_based_recommender' (c:\Users\Ridmi\Documents\movie-recommender\notebooks\../src\content_based_recommender.py)

In [1]:
import pandas as pd
import sys
sys.path.append('../src')

from scipy.sparse import load_npz
from content_based_recommender import (
    build_combined_similarity,
    recommend_similar_movies
)

# Load data
movies = pd.read_csv('../data/processed/movies_clean.csv')
genres_encoded = pd.read_csv('../data/features/genres_encoded.csv')
tfidf_matrix = load_npz('../data/features/tfidf_matrix.npz')
tfidf_movie_ids = pd.read_csv('../data/features/tfidf_movieId_index.csv')['movieId']

# Build combined similarity
print("Building combined similarity matrix — may take 1-3 minutes...")
combined_similarity = build_combined_similarity(
    genres_df=genres_encoded,
    tfidf_matrix=tfidf_matrix,
    tfidf_movie_ids=tfidf_movie_ids,
    genre_weight=0.3,
    tag_weight=0.7
)
print("Done. Shape:", combined_similarity.shape)

# Sanity check
print("\nSanity check:")
print("Toy Story vs Toy Story 2:", round(combined_similarity.loc[1, 3114], 4))
print("Toy Story vs Magic Crystal:", round(combined_similarity.loc[1, 117454], 4))
print("(Toy Story 2 should score noticeably higher than Magic Crystal)")

# Recommend
print("\nTop 10 movies similar to Toy Story (combined genres + tags):")
print(recommend_similar_movies(1, combined_similarity, movies, n=10))

Building combined similarity matrix — may take 1-3 minutes...
Movies with both genre and tag data: 19,545
Done. Shape: (27278, 27278)

Sanity check:
Toy Story vs Toy Story 2: 0.9588
Toy Story vs Magic Crystal: 0.3
(Toy Story 2 should score noticeably higher than Magic Crystal)

Top 10 movies similar to Toy Story (combined genres + tags):
   movieId                       title  \
0     3114          Toy Story 2 (1999)   
1     4886       Monsters, Inc. (2001)   
2     2355        Bug's Life, A (1998)   
3     5218              Ice Age (2002)   
4     6377         Finding Nemo (2003)   
5    78499          Toy Story 3 (2010)   
6     2294                 Antz (1998)   
7     8961     Incredibles, The (2004)   
8   103141  Monsters University (2013)   
9    50872          Ratatouille (2007)   

                                             genres  similarity_score  
0       Adventure|Animation|Children|Comedy|Fantasy          0.958827  
1       Adventure|Animation|Children|Comedy|Fantasy  

In [2]:
# Test with a Drama/Thriller - Inception (movieId = 4226)
print("Movies similar to Inception:")
print(recommend_similar_movies(4226, combined_similarity, movies, n=10))

# Test with a classic - The Godfather (movieId = 858)
print("\nMovies similar to The Godfather:")
print(recommend_similar_movies(858, combined_similarity, movies, n=10))

Movies similar to Inception:
   movieId                                           title  \
0     1625                                Game, The (1997)   
1     8950          Machinist, The (Maquinista, El) (2004)   
2    26875  Pure Formality, A (Pura formalità, Una) (1994)   
3     8783                             Village, The (2004)   
4    27773                                  Old Boy (2003)   
5   104552                               Crawlspace (2012)   
6    48780                            Prestige, The (2006)   
7   103235   Best Offer, The (Migliore offerta, La) (2013)   
8    65882                           Uninvited, The (2009)   
9   100383                             Side Effects (2013)   

                             genres  similarity_score  
0            Drama|Mystery|Thriller          0.669069  
1                  Mystery|Thriller          0.667767  
2  Crime|Film-Noir|Mystery|Thriller          0.621032  
3            Drama|Mystery|Thriller          0.610819  
4       

In [3]:
import numpy as np
import pandas as pd
import joblib
import sys
sys.path.append('../src')

from scipy.sparse import load_npz
from collaborative_filtering import train_svd, recommend_for_user

# Load everything
sparse_user_item = load_npz('../data/features/sparse_user_item.npz')
user_to_idx = joblib.load('../data/features/user_to_idx.pkl')
idx_to_movie = joblib.load('../data/features/idx_to_movie.pkl')
ratings = pd.read_csv('../data/processed/ratings_clean.csv')
movies = pd.read_csv('../data/processed/movies_clean.csv')

print("Sparse matrix shape:", sparse_user_item.shape)

# Train SVD — the actual learning step
print("Training SVD (50 latent factors)...")
svd, user_factors = train_svd(sparse_user_item, n_components=50)
print(f"Done. Variance explained: {svd.explained_variance_ratio_.sum()*100:.2f}%")
print(f"user_factors shape: {user_factors.shape}")

# Recommend for User 1 — memory safe, no full matrix needed
print("\nTop 10 recommendations for User 1:")
recs = recommend_for_user(
    user_id=1,
    svd=svd,
    user_factors=user_factors,
    user_to_idx=user_to_idx,
    idx_to_movie=idx_to_movie,
    ratings=ratings,
    movies=movies,
    n=10
)
print(recs)

Sparse matrix shape: (138493, 26744)
Training SVD (50 latent factors)...
Done. Variance explained: 36.58%
user_factors shape: (138493, 50)

Top 10 recommendations for User 1:
   movieId                                              title  \
0     1197                         Princess Bride, The (1987)   
1     1073         Willy Wonka & the Chocolate Factory (1971)   
2     2115        Indiana Jones and the Temple of Doom (1984)   
3     1206                         Clockwork Orange, A (1971)   
4     2571                                 Matrix, The (1999)   
5      608                                       Fargo (1996)   
6      551             Nightmare Before Christmas, The (1993)   
7     5618  Spirited Away (Sen to Chihiro no kamikakushi) ...   
8     1210  Star Wars: Episode VI - Return of the Jedi (1983)   
9     3793                                       X-Men (2000)   

                                    genres  predicted_rating  
0  Action|Adventure|Comedy|Fantasy|Romance    

In [4]:
# Recommend for a different user — should produce a different list
print("Top 10 recommendations for User 500:")
recs_500 = recommend_for_user(
    user_id=500,
    svd=svd,
    user_factors=user_factors,
    user_to_idx=user_to_idx,
    idx_to_movie=idx_to_movie,
    ratings=ratings,
    movies=movies,
    n=10
)
print(recs_500)

Top 10 recommendations for User 500:
   movieId                                              title  \
0     2329                          American History X (1998)   
1     2019        Seven Samurai (Shichinin no samurai) (1954)   
2     1206                         Clockwork Orange, A (1971)   
3      908                          North by Northwest (1959)   
4     1252                                   Chinatown (1974)   
5    79132                                   Inception (2010)   
6      924                       2001: A Space Odyssey (1968)   
7      919                           Wizard of Oz, The (1939)   
8     1250               Bridge on the River Kwai, The (1957)   
9     1201  Good, the Bad and the Ugly, The (Buono, il bru...   

                                            genres  predicted_rating  
0                                      Crime|Drama          3.647034  
1                           Action|Adventure|Drama          2.884656  
2                      Crime|Drama